# Neural forecasting — redesigned paper results

This notebook produces the agreed scientific figure, quality–NFE and quality–runtime curves, a **25-row appendix table**, and a compact diagnostic figure. It uses the completed formal experiment with six recordings and three training runs. It **does not train, modify the model or loss, change the formal configuration, or select new checkpoints**.

**Run it:** put this file in the project's `notebooks/` directory, use the environment of your completed neural experiment, restart the kernel and **Run All**. A GPU is recommended for the first pass. New scores evaluate NFE **1, 2, 4, 8, 16, 32, 64** for Count Flow Map and both Count-FM samplers. The table also preserves Count-FM at **128 and 256**. Existing scores are reused. Only missing budgets are evaluated, on the original test histories with 64 draws. Every latency is remeasured using the same machine and timing protocol. Run without competing GPU jobs.

**Outputs:** `outputs/neural_paper_v2/manuscript_results_v3/`. The existing `manuscript_population/population_cache/` is reused for the high-activity analysis. Original model checkpoints, metric files, and training configuration remain untouched. A deleted population cache is regenerated from its frozen checkpoint. New `nfe_cache/` files contain per-history scores and RNG states, not large neuron-by-neuron sample tensors. `timing_cache/` contains small timing records.

**Resume:** interrupt and rerun the notebook. Missing-budget inference saves every 256 histories and restores the random state. Timing is saved after each operating point. On an abrupt kill, at most the unsaved block or current timing measurement is repeated. Resume partial score files on the same software/device. A different timing environment gets its own timing cache. Keep the original data and the selected checkpoints named in `complete.json`.

No example results, simulated scores or pre-executed outputs are embedded. The newly evaluated curves must be inspected before writing performance claims.

In [ ]:
from pathlib import Path
from contextlib import contextmanager
import hashlib
import inspect
import itertools
import json
import os
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'outputs/neural_paper_v2/config_snapshot.json').is_file()
             and (p / 'countflow/neural_models.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the project containing outputs/neural_paper_v2.')
sys.path.insert(0, str(ROOT))
from countflow.neural_data import SpikeSession, fingerprint, file_hash
from countflow.neural_models import build_model, forecast
from countflow.neural_evaluation import isolated_rng, shuffle_neurons
from countflow.neural_experiment import evaluation_source_signature
from countflow.neural_training import load_checkpoint

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)
SOURCE = ROOT / 'outputs/neural_paper_v2'
PAPER = SOURCE / 'manuscript_results_v3'
CACHE = SOURCE / 'manuscript_population' / 'population_cache'
if PAPER.is_symlink() or PAPER.resolve().parent != SOURCE.resolve():
    raise ValueError('The presentation destination must be an ordinary subdirectory of the existing output.')

def require(condition, message):
    if not condition:
        raise ValueError(message)

INPUTS = ['config_snapshot.json', 'method_protocol.json', 'report_status.json',
          'all_metrics.csv', 'rollout_metrics.csv', 'paper_neural_matched_nfe.csv']
for name in INPUTS:
    if not (SOURCE / name).is_file():
        raise FileNotFoundError(f'Missing formal result: {SOURCE / name}')
INPUT_HASHES = {name: file_hash(SOURCE / name) for name in INPUTS}
config, protocol, status = [json.loads((SOURCE / name).read_text()) for name in INPUTS[:3]]
require(config.get('purpose') == 'paper', 'Diagnostic runs cannot produce these paper outputs.')
require(status.get('status') == 'complete' and not status.get('missing'), 'The formal experiment is incomplete.')
require(config['sessions'] == [4, 7, 11, 18, 25, 33] and config['seeds'] == [42, 123, 2026],
        'Expected the complete six-recording, three-seed experiment.')
require(status.get('n_sessions') == 6 and status.get('n_seeds') == 3, 'Replication counts disagree.')
require(protocol.get('method_protocol') == 'diagonal_ck_online_absolute_v1'
        and protocol.get('purpose') == 'paper'
        and all(status.get(k) == v for k, v in protocol.items()), 'The report uses a different method protocol.')
require(config['training']['steps'] == 30000
        and config['training'].get('endpoint_weight', 0) == 0
        and config['training'].get('ck_teacher') == 'online_stop_gradient'
        and config['model'].get('source_conditioned_support', False) is False
        and config['model'].get('correction_time_scale') == 'absolute',
        'The saved experiment does not use the manuscript loss and kernel.')
e = config['evaluation']
require(e['draws'] == 64 and e['cfm_nfe'] == [1, 16] and e['test_cases'] == 4096
        and config['history_bins'] == 10 and e['rollout_horizons'] == [1, 2, 5, 10],
        'The evaluation settings differ from the agreed formal protocol.')
metrics = pd.read_csv(SOURCE / 'all_metrics.csv')
rollouts = pd.read_csv(SOURCE / 'rollout_metrics.csv')
stored_table = pd.read_csv(SOURCE / 'paper_neural_matched_nfe.csv')
print('Device:', DEVICE)
print('Existing formal experiment:', SOURCE)
print('New analysis and paper files:', PAPER)
print('The first run adds missing-budget inference and consistent timing; completed caches are reused.')


## Verify the existing formal results

The original 1- and 16-NFE table must agree with its underlying results. All six recordings and three training runs are required. Throughout this notebook, recordings receive equal weight within each seed, followed by the mean and sample SD across the three seed averages. This SD measures variation across training runs, not biological-replicate uncertainty.

In [ ]:
SCORES = ['energy_score', 'population_crps', 'neuron_rmse', 'ensemble_ms']
LEVELS = [50, 80, 90, 95]
COVERAGE = [f'coverage_{x}' for x in LEVELS]
KEY = ['session', 'seed', 'method', 'sampler', 'nfe']
PAIRS = set(itertools.product(config['sessions'], config['seeds']))
LABELS = ['Count Flow Map (1 NFE)', 'Count Flow Map (16 NFE)',
          'Count-FM (unit, 1 NFE)', 'Count-FM (unit, 16 NFE)',
          'Count-FM (tau, 1 NFE)', 'Count-FM (tau, 16 NFE)']
flow = metrics[metrics.method.isin(['cfm', 'fm']) & metrics.nfe.isin([1, 16])].copy()
expected = {(s, seed, method, sampler, nfe) for s, seed in PAIRS
            for method, sampler in [('cfm', 'map'), ('fm', 'unit'), ('fm', 'tau')] for nfe in [1, 16]}
require(not flow.duplicated(KEY).any() and set(flow[KEY].itertuples(index=False, name=None)) == expected,
        'A matched-budget result is missing or duplicated.')
require(np.isfinite(flow[SCORES + COVERAGE]).all().all() and flow.ensemble_ms.gt(0).all(),
        'Invalid formal scores or latency.')
require(flow.completed_steps.eq(30000).all(), 'A formal fit is incomplete.')
flow['label'] = flow.apply(lambda r: f'Count Flow Map ({r.nfe} NFE)' if r.method == 'cfm'
                          else f'Count-FM ({r.sampler}, {r.nfe} NFE)', axis=1)

def summarize(frame, group, columns):
    seeds = frame.groupby(group + ['seed'], sort=False)[columns].mean().reset_index()
    result = seeds.groupby(group, sort=False)[columns].agg(['mean', 'std'])
    result.columns = [f'{a}_{b}' for a, b in result.columns]
    return result.reset_index(), seeds

summary, table_seed_means = summarize(flow, ['label'], SCORES)
summary = summary.set_index('label').loc[LABELS].reset_index()
columns = [f'{m}_{s}' for m in SCORES for s in ['mean', 'std']]
require(not stored_table.label.duplicated().any() and set(stored_table.label) == set(LABELS),
        'The saved matched-budget table has unexpected rows.')
require(np.allclose(summary[columns], stored_table.set_index('label').loc[LABELS, columns],
                    rtol=1e-10, atol=1e-12), 'The formal table does not agree with its underlying scores.')

DIGITS = dict(energy_score=4, population_crps=5, neuron_rmse=4, ensemble_ms=2)
HEADINGS = ['Energy score ↓', 'Population CRPS ↓', 'RMSE ↓', 'Latency (ms) ↓']
METHOD_NAMES = ['Count Flow Map', 'Count Flow Map', 'Count-FM (unit jump)',
                'Count-FM (unit jump)', 'Count-FM (binomial tau-leap)', 'Count-FM (binomial tau-leap)']
table_display = pd.DataFrame({'Method': METHOD_NAMES, 'NFE': [1, 16, 1, 16, 1, 16]})
for metric, heading in zip(SCORES, HEADINGS):
    n = DIGITS[metric]
    table_display[heading] = [f'{r[metric + "_mean"]:.{n}f} ± {r[metric + "_std"]:.{n}f}'
                              for _, r in summary.iterrows()]
print('Verified the original six-row comparison and all replication identifiers.')


## Fixed models and resumable evaluation helpers

Both Count-FM samplers share their selected rate network. Count Flow Map uses one selected checkpoint across all budgets. The notebook reads those paths from the completed formal evaluations. The additional grid is a reporting sweep and performs no validation-based reselection.

High activity is defined by a strict exceedance of the recording's training-set 95th percentile of total population counts. The Brier score uses the exceedance fraction among 64 predictive samples. Independently permuting draws across neurons within a history preserves every neuronal empirical marginal. We report the shuffled minus joint Brier score, retaining either sign.

In [ ]:
QUANTILES = np.array([.50, .60, .70, .80, .85, .90, .95, .975, .99])
EVENT_COLUMN = int(np.flatnonzero(QUANTILES == .95)[0])
SAVE_EVERY_CASES = 256
ANALYSIS_VERSION = 'population_activity_v1'

def contained_path(base, relative):
    path = (base / relative).resolve()
    require(path.is_relative_to(base.resolve()), 'A saved result points outside the experiment directory.')
    return path

def atomic_npz(path, **values):
    temporary = path.with_name(path.name + f'.{os.getpid()}.tmp')
    try:
        with temporary.open('wb') as stream:
            np.savez_compressed(stream, **values)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

@contextmanager
def analysis_lock():
    # Released by the OS if the HPC job or kernel is terminated.
    import fcntl
    CACHE.mkdir(parents=True, exist_ok=True)
    with (CACHE / '.analysis.lock').open('a') as handle:
        try:
            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError('Another notebook is writing this population-analysis cache.') from error
        try:
            yield
        finally:
            fcntl.flock(handle, fcntl.LOCK_UN)

def read_cache(path):
    with np.load(path, allow_pickle=False) as saved:
        return {name: saved[name].copy() for name in saved.files}

def cache_contract(session, seed, nfe, indices, checkpoint_hash, complete, thresholds):
    return {'version': ANALYSIS_VERSION, 'purpose': 'paper', 'session': session.index,
            'seed': seed, 'nfe': nfe, 'draws': e['draws'], 'case_batch': e['case_batch'],
            'generation_seed': e['seed'], 'shuffle_seed': e['seed'] + 1,
            'data_signature': session.signature, 'checkpoint_sha256': checkpoint_hash,
            'evaluation_signature': complete['signature'], 'protocol': protocol,
            'indices_sha256': hashlib.sha256(np.asarray(indices, dtype=np.int64).tobytes()).hexdigest(),
            'training_quantiles': QUANTILES.tolist(), 'total_count_thresholds': thresholds.tolist(),
            'event_comparison': 'strictly_greater', 'implementation_sha256': ANALYSIS_CODE_HASH}

@torch.no_grad()
def population_forecasts(model, session, seed, nfe, indices, contract, path, device):
    """One resumable, compact cache per recording, training seed and NFE."""
    total, draws, batch_size = len(indices), e['draws'], e['case_batch']
    key = fingerprint(contract)
    runtime = {'torch': str(torch.__version__), 'numpy': np.__version__,
               'device_type': torch.device(device).type,
               'device_name': torch.cuda.get_device_name(device) if torch.device(device).type == 'cuda' else platform.machine()}
    done = 0
    joint = np.empty((total, draws), dtype=np.int64)
    shuffled = np.empty_like(joint)
    saved = None
    if path.exists():
        saved = read_cache(path)
        require(str(saved['signature']) == key, f'Stale population cache: {path}. Keep it separate from this run.')
        require(np.array_equal(saved['indices'], indices), 'Cached test indices disagree.')
        done = int(saved['done'])
        require(0 <= done <= total and saved['joint'].shape == saved['shuffled'].shape == (done, draws)
                and (saved['joint'] >= 0).all() and (saved['shuffled'] >= 0).all(), 'Invalid population cache.')
        joint[:done], shuffled[:done] = saved['joint'], saved['shuffled']
        if done == total:
            print(f'[session={session.index:03d} seed={seed} NFE={nfe}] reuse {done} histories', flush=True)
            return joint, shuffled
        require(json.loads(str(saved['runtime'])) == runtime,
                f'Partial cache was generated on different software/device: {path}. Resume on the original environment or remove only this partial analysis file.')
        require(done % batch_size == 0, 'A partial cache ends inside an inference batch.')
    shuffle_rng = np.random.default_rng(e['seed'] + 1)
    model.eval()
    with isolated_rng(e['seed'], device):
        if saved is not None:
            torch.set_rng_state(torch.from_numpy(saved['torch_rng']))
            if torch.device(device).type == 'cuda':
                torch.cuda.set_rng_state(torch.from_numpy(saved['cuda_rng']), device)
            shuffle_rng.bit_generator.state = json.loads(str(saved['shuffle_rng']))
        committed = done
        for first in range(done, total, batch_size):
            last = min(first + batch_size, total)
            history, _ = session.batch(indices[first:last], device, split='test')
            counts = forecast(model, 'cfm', history, draws, nfe, 'map', config['training']['tau'])
            require(counts.shape == (last-first, draws, session.dim) and torch.isfinite(counts).all().item()
                    and (counts >= 0).all().item() and (counts == counts.round()).all().item(),
                    'The model returned invalid spike counts.')
            joint[first:last] = counts.to(torch.int64).sum(-1).cpu().numpy()
            control = shuffle_neurons(counts, shuffle_rng)
            shuffled[first:last] = control.to(torch.int64).sum(-1).cpu().numpy()
            if last - committed >= SAVE_EVERY_CASES or last == total:
                atomic_npz(path, signature=np.array(key), contract=np.array(json.dumps(contract)),
                           runtime=np.array(json.dumps(runtime)), indices=indices, done=np.array(last),
                           joint=joint[:last], shuffled=shuffled[:last],
                           torch_rng=torch.get_rng_state().cpu().numpy(),
                           cuda_rng=torch.cuda.get_rng_state(device).cpu().numpy()
                           if torch.device(device).type == 'cuda' else np.empty(0, dtype=np.uint8),
                           shuffle_rng=np.array(json.dumps(shuffle_rng.bit_generator.state)))
                committed = last
                print(f'[session={session.index:03d} seed={seed} NFE={nfe}] {last}/{total} histories saved', flush=True)
    return joint, shuffled

def population_statistics(joint, shuffled, observed_total, thresholds):
    rows = []
    for quantile, threshold in zip(QUANTILES, thresholds):
        rows.append({'training_percentile': 100 * quantile, 'threshold_total_count': threshold,
                     'observed': float((observed_total > threshold).mean()),
                     'joint': float((joint > threshold).mean()),
                     'shuffled': float((shuffled > threshold).mean())})
    threshold = thresholds[EVENT_COLUMN]
    outcome = (observed_total > threshold).astype(float)
    p = (joint > threshold).mean(1)
    p_shuffle = (shuffled > threshold).mean(1)
    brier = np.square(p-outcome)
    brier_shuffle = np.square(p_shuffle-outcome)
    return rows, {'observed_event_rate': outcome.mean(), 'joint_event_probability': p.mean(),
                  'shuffled_event_probability': p_shuffle.mean(), 'brier_joint': brier.mean(),
                  'brier_shuffled': brier_shuffle.mean(), 'brier_difference': (brier_shuffle-brier).mean()}, \
           {'observed_event': outcome.astype(np.int8), 'joint_probability': p,
            'shuffled_probability': p_shuffle, 'brier_joint': brier, 'brier_shuffled': brier_shuffle}

ANALYSIS_CODE_HASH = hashlib.sha256((ANALYSIS_VERSION + '\n' + inspect.getsource(shuffle_neurons)).encode()).hexdigest()


In [ ]:
# Fixed reporting grid. These settings do not modify the formal experiment config.
COMMON_NFE = [1, 2, 4, 8, 16, 32, 64]
FM_EXTRA_NFE = [128, 256]
VARIANTS = [('cfm', 'map', n) for n in COMMON_NFE] + [
    ('fm', sampler, n) for sampler in ['unit', 'tau'] for n in COMMON_NFE + FM_EXTRA_NFE]
SWEEP_VERSION = 'frozen_checkpoint_nfe_sweep_v1'
SWEEP_CACHE = PAPER / 'nfe_cache'
from countflow.neural_evaluation import sample_scores, combine_scores
from countflow.neural_experiment import ensemble_latency
from countflow.neural_training import atomic_json

def runtime_info(device):
    target = torch.device(device)
    result = {'torch': str(torch.__version__), 'numpy': np.__version__,
              'device_type': target.type, 'cpu_threads': torch.get_num_threads(),
              'float32_matmul_precision': torch.get_float32_matmul_precision(),
              'tf32_matmul': torch.backends.cuda.matmul.allow_tf32,
              'tf32_cudnn': torch.backends.cudnn.allow_tf32,
              'cudnn_benchmark': torch.backends.cudnn.benchmark,
              'deterministic_algorithms': torch.are_deterministic_algorithms_enabled()}
    if target.type == 'cuda':
        prop = torch.cuda.get_device_properties(target)
        result.update(device_name=prop.name, cuda=str(torch.version.cuda),
                      cudnn=torch.backends.cudnn.version(), capability=[prop.major, prop.minor],
                      multiprocessors=prop.multi_processor_count, total_memory=prop.total_memory)
    else:
        result.update(device_name=platform.processor() or platform.machine(), host=platform.node())
    return result

TIMING_RUNTIME = runtime_info(DEVICE)
TIMING_ID = fingerprint(TIMING_RUNTIME)[:16]
TIMING_CACHE = PAPER / 'timing_cache' / TIMING_ID
SCORE_IMPLEMENTATION = hashlib.sha256((SWEEP_VERSION + inspect.getsource(sample_scores)
                                        + inspect.getsource(combine_scores)).encode()).hexdigest()
TIMING_IMPLEMENTATION = hashlib.sha256(inspect.getsource(ensemble_latency).encode()).hexdigest()

@contextmanager
def sweep_lock():
    import fcntl
    PAPER.mkdir(parents=True, exist_ok=True)
    with (PAPER / '.sweep.lock').open('a') as handle:
        try:
            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError('Another process is evaluating this paper sweep.') from error
        try:
            yield
        finally:
            fcntl.flock(handle, fcntl.LOCK_UN)

def validate_session(session):
    manifest = SOURCE / 'data_manifest' / f'session_{session.index:03d}.json'
    require(session.signature == json.loads(manifest.read_text())['signature'],
            f'Spike data differ from the formal run: recording {session.index}.')

def frozen_fit(session, seed, method):
    directory = SOURCE / 'evaluations' / f'session_{session.index:03d}' / f'seed_{seed}' / method
    complete = json.loads((directory / 'complete.json').read_text())
    path = contained_path(SOURCE, complete['selected_checkpoint'])
    checkpoint_hash = file_hash(path)
    signature = fingerprint({'checkpoint': checkpoint_hash, 'evaluation': e,
                             'source': evaluation_source_signature(method), 'data': session.signature})
    require(complete['signature'] == signature,
            f'Formal data, checkpoint or evaluation source mismatch: {directory}')
    selected = load_checkpoint(path)
    require(selected['completed_steps'] == config['training']['steps'], 'Incomplete selected model.')
    rows = metrics[(metrics.session == session.index) & (metrics.seed == seed) & (metrics.method == method)]
    require(len(rows) > 0 and rows.selected_update.eq(selected['best_step']).all()
            and rows.candidate_index.nunique() == 1 and rows.n_neurons.eq(session.dim).all(),
            'Formal results do not share the frozen selected checkpoint.')
    metadata = {k: int(rows.iloc[0][k]) for k in
                ['session', 'seed', 'selected_update', 'candidate_index', 'completed_steps', 'n_neurons']}
    metadata['method'] = method
    return directory, complete, path, checkpoint_hash, selected, metadata

def score_contract(session, seed, method, sampler, nfe, indices, complete, checkpoint_hash):
    return {'version': SWEEP_VERSION, 'purpose': 'paper', 'session': session.index, 'seed': seed,
            'method': method, 'sampler': sampler, 'nfe': nfe, 'draws': e['draws'],
            'tau': config['training']['tau'], 'case_batch': e['case_batch'], 'generation_seed': e['seed'],
            'data_signature': session.signature, 'checkpoint_sha256': checkpoint_hash,
            'formal_signature': complete['signature'], 'implementation_sha256': SCORE_IMPLEMENTATION,
            'indices_sha256': hashlib.sha256(indices.astype(np.int64).tobytes()).hexdigest()}

def scores_from_arrays(arrays):
    return combine_scores([arrays])[0]

@torch.no_grad()
def evaluate_missing(model, session, method, sampler, nfe, indices, contract, path, device):
    """Use the formal scorer and batch order, saving small per-history scores and RNG state."""
    signature = fingerprint(contract)
    total, done = len(indices), 0
    arrays, saved = {}, None
    current_runtime = runtime_info(device)
    if path.exists():
        saved = read_cache(path)
        require(str(saved['signature']) == signature and np.array_equal(saved['indices'], indices),
                f'NFE cache provenance mismatch: {path}')
        done = int(saved['done'])
        arrays = {key[6:]: value for key, value in saved.items() if key.startswith('score_')}
        require(0 <= done <= total and bool(arrays) and all(
            value.shape == (done,) and np.isfinite(value).all() for value in arrays.values()),
            'Invalid cached score arrays.')
        if done == total:
            return scores_from_arrays(arrays), 'cached_sweep'
        require(done % e['case_batch'] == 0 and json.loads(str(saved['runtime'])) == current_runtime,
                f'Resume this partial cache on the same software/device: {path}')
    scorer_rng = np.random.default_rng(e['seed'])
    model.eval()
    with isolated_rng(e['seed'], device):
        if saved is not None:
            torch.set_rng_state(torch.from_numpy(saved['torch_rng']))
            if torch.device(device).type == 'cuda':
                torch.cuda.set_rng_state(torch.from_numpy(saved['cuda_rng']), device)
            scorer_rng.bit_generator.state = json.loads(str(saved['scorer_rng_state']))
        parts = [arrays] if done else []
        committed = done
        for first in range(done, total, e['case_batch']):
            last = min(first + e['case_batch'], total)
            history, observed = session.batch(indices[first:last], device, split='test')
            samples = forecast(model, method, history, e['draws'], nfe, sampler, config['training']['tau'])
            require(samples.shape == (last-first, e['draws'], session.dim)
                    and torch.isfinite(samples).all().item() and (samples >= 0).all().item()
                    and (samples == samples.round()).all().item(), 'Invalid generated counts.')
            parts.append(sample_scores(samples, observed[:, 0], scorer_rng))
            if last - committed >= SAVE_EVERY_CASES or last == total:
                arrays = {key: np.concatenate([p[key] for p in parts]) for key in parts[0]}
                require(all(np.isfinite(v).all() for v in arrays.values()), 'Nonfinite forecast score.')
                atomic_npz(path, signature=np.array(signature), contract=np.array(json.dumps(contract)),
                           runtime=np.array(json.dumps(current_runtime)), indices=indices, done=np.array(last),
                           torch_rng=torch.get_rng_state().cpu().numpy(),
                           cuda_rng=torch.cuda.get_rng_state(device).cpu().numpy()
                           if torch.device(device).type == 'cuda' else np.empty(0, dtype=np.uint8),
                           scorer_rng_state=np.array(json.dumps(scorer_rng.bit_generator.state)),
                           **{'score_'+k: v for k, v in arrays.items()})
                parts, committed = [arrays], last
                print(f'  {method}/{sampler} NFE={nfe}: {last}/{total} histories saved', flush=True)
    return scores_from_arrays(arrays), 'new_sweep'

def formal_scores(directory, complete, metadata, sampler, nfe, indices):
    selected_rows = metrics[(metrics.session == metadata['session']) & (metrics.seed == metadata['seed'])
                            & (metrics.method == metadata['method']) & (metrics.sampler == sampler)
                            & (metrics.nfe == nfe)]
    if selected_rows.empty:
        return None
    require(len(selected_rows) == 1, 'Duplicated formal evaluation row.')
    path = directory / f'{sampler}_{nfe}.pt'
    stored = load_checkpoint(path)
    require(stored['signature'] == complete['signature']
            and np.array_equal(stored['test']['target_indices'], indices), 'Stale formal score cache.')
    row = stored['row']
    require(all(row[k] == v for k, v in metadata.items())
            and row['nfe'] == nfe and row['sampler'] == sampler, 'Formal checkpoint metadata disagree.')
    compare = SCORES + COVERAGE
    require(all(np.isclose(row[k], selected_rows.iloc[0][k], rtol=1e-10, atol=1e-12) for k in compare),
            'Formal CSV and evaluation cache disagree.')
    return {k: float(row[k]) for k in SCORES[:-1] + COVERAGE}

def measured_latency(model, session, method, sampler, nfe, indices, contract, path):
    timing_contract = {**contract, 'runtime': TIMING_RUNTIME, 'timing_cases': e['timing_cases'],
                       'timing_repeats': e['timing_repeats'], 'timing_seed': e['seed'] + 777,
                       'timing_implementation': TIMING_IMPLEMENTATION}
    signature = fingerprint(timing_contract)
    if path.exists():
        stored = json.loads(path.read_text())
        require(stored['signature'] == signature, f'Timing cache provenance mismatch: {path}')
        value = stored['ensemble_ms']
    else:
        value = ensemble_latency(model, method, session, config, indices, DEVICE, nfe, sampler)
        require(np.isfinite(value) and value > 0, 'Invalid measured latency.')
        atomic_json({'signature': signature, 'contract': timing_contract, 'ensemble_ms': value}, path)
    require(np.isfinite(value) and value > 0, 'Invalid cached latency.')
    return float(value)

def run_nfe_sweep():
    results, provenance = [], []
    with sweep_lock():
        SWEEP_CACHE.mkdir(parents=True, exist_ok=True)
        TIMING_CACHE.mkdir(parents=True, exist_ok=True)
        for session_id in config['sessions']:
            session = SpikeSession(ROOT / config['data_root'], session_id, config['history_bins'])
            validate_session(session)
            indices = session.indices('test', e['test_cases'])
            for seed in config['seeds']:
                for method in ['cfm', 'fm']:
                    directory, complete, checkpoint, checkpoint_hash, selected, metadata = frozen_fit(session, seed, method)
                    model = None
                    for this_method, sampler, nfe in VARIANTS:
                        if this_method != method:
                            continue
                        tag = f'session_{session_id:03d}_seed_{seed}_{method}_{sampler}_{nfe}'
                        print(f'[{len(results)+1}/{len(VARIANTS)*len(PAIRS)}] {tag}', flush=True)
                        contract = score_contract(session, seed, method, sampler, nfe, indices, complete, checkpoint_hash)
                        score_path, timing_path = SWEEP_CACHE / (tag+'.npz'), TIMING_CACHE / (tag+'.json')
                        scores = formal_scores(directory, complete, metadata, sampler, nfe, indices)
                        score_origin = 'formal_evaluation'
                        # Loading a completed cache does not call the sampler or timing function.
                        if scores is None or not timing_path.exists():
                            if model is None:
                                model = build_model(method, session, config).to(DEVICE).eval()
                                model.load_state_dict(selected['model'], strict=True)
                        if scores is None:
                            scores, score_origin = evaluate_missing(model, session, method, sampler, nfe,
                                                                    indices, contract, score_path, DEVICE)
                        latency = measured_latency(model, session, method, sampler, nfe, indices, contract, timing_path)
                        results.append({**metadata, 'sampler': sampler, 'nfe': nfe,
                                        **{k: scores[k] for k in SCORES[:-1] + COVERAGE},
                                        'ensemble_ms': latency, 'score_source': score_origin})
                        provenance.append({**contract, 'selected_checkpoint': str(checkpoint.relative_to(SOURCE)),
                                           'score_source': score_origin,
                                           'score_file': str((directory / f'{sampler}_{nfe}.pt') if score_origin == 'formal_evaluation' else score_path),
                                           'timing_file': str(timing_path)})
                    del model, selected
    result = pd.DataFrame(results)
    expected = {(s, seed, m, sampler, n) for s, seed in PAIRS for m, sampler, n in VARIANTS}
    require(not result.duplicated(KEY).any() and set(result[KEY].itertuples(index=False, name=None)) == expected,
            'The full NFE sweep is incomplete.')
    require(np.isfinite(result[SCORES + COVERAGE]).all().all(), 'Nonfinite sweep output.')
    return result, provenance

print(f'Full table: {len(VARIANTS)} operating points × {len(PAIRS)} recording/run pairs.')
print('Existing formal scores are reused; only missing budgets generate new score samples.')
print('Every latency is measured with the same runtime profile:', TIMING_RUNTIME)


## Reuse the completed population analysis

This loads the existing compact caches at 1 and 16 NFE, verifying data and checkpoint provenance. It generates forecasts only for missing or partially completed population caches. This step is separate from the new NFE sweep.

In [ ]:
def run_population_analysis():
    distributions, events, event_cases, provenance, recording_info = [], [], [], [], []
    current_source = evaluation_source_signature('cfm')
    with analysis_lock():
        for session_id in config['sessions']:
            session = SpikeSession(ROOT / config['data_root'], session_id, config['history_bins'])
            data_manifest = json.loads((SOURCE / 'data_manifest' / f'session_{session_id:03d}.json').read_text())
            require(session.signature == data_manifest['signature'], 'Spike data differ from the formal experiment.')
            # One integer total per time bin; never pool neurons across recordings.
            population_total = np.asarray(session.counts.sum(axis=0, dtype=np.int64))
            train_start, train_end = session.bounds['train']
            thresholds = np.quantile(population_total[train_start:train_end], QUANTILES, method='linear')
            indices = session.indices('test', e['test_cases'])
            observed = population_total[indices]
            recording_info.append({'session': session_id, 'n_neurons': session.dim, 'test_histories': len(indices),
                                   'event_threshold_total': thresholds[EVENT_COLUMN],
                                   'event_threshold_mean_count': thresholds[EVENT_COLUMN]/session.dim,
                                   'observed_event_rate': float((observed > thresholds[EVENT_COLUMN]).mean())})
            for seed in config['seeds']:
                directory = SOURCE / 'evaluations' / f'session_{session_id:03d}' / f'seed_{seed}' / 'cfm'
                complete = json.loads((directory / 'complete.json').read_text())
                checkpoint_path = contained_path(SOURCE, complete['selected_checkpoint'])
                checkpoint_hash = file_hash(checkpoint_path)
                expected_signature = fingerprint({'checkpoint': checkpoint_hash, 'evaluation': e,
                                                  'source': current_source, 'data': session.signature})
                require(complete['signature'] == expected_signature,
                        f'Checkpoint, data, or forecasting source differs from the completed formal run: {directory}')
                selected = load_checkpoint(checkpoint_path)
                require(selected['completed_steps'] == config['training']['steps'], 'Incomplete selected checkpoint.')
                model = build_model('cfm', session, config).to(DEVICE).eval()
                model.load_state_dict(selected['model'], strict=True)
                for nfe in [1, 16]:
                    stored = load_checkpoint(directory / f'map_{nfe}.pt')
                    require(stored['signature'] == complete['signature'], 'Stale saved evaluation.')
                    require(np.array_equal(stored['test']['target_indices'], indices), 'Saved test histories disagree.')
                    row = stored['row']
                    require(row['session'] == session_id and row['seed'] == seed and row['nfe'] == nfe
                            and row['method'] == 'cfm' and row['sampler'] == 'map'
                            and row['selected_update'] == selected['best_step'], 'Selected model metadata disagree.')
                    original = flow[(flow.session == session_id) & (flow.seed == seed)
                                    & (flow.method == 'cfm') & (flow.nfe == nfe)].iloc[0]
                    require(all(np.isclose(original[k], row[k], rtol=1e-10, atol=1e-12) for k in SCORES)
                            and original.candidate_index == row['candidate_index'], 'Saved metric sources disagree.')
                    contract = cache_contract(session, seed, nfe, indices, checkpoint_hash, complete, thresholds)
                    path = CACHE / f'session_{session_id:03d}_seed_{seed}_nfe_{nfe}.npz'
                    joint, shuffled = population_forecasts(model, session, seed, nfe, indices, contract, path, DEVICE)
                    curves, scores, cases = population_statistics(joint, shuffled, observed, thresholds)
                    identity = {'session': session_id, 'seed': seed, 'nfe': nfe}
                    distributions.extend([{**identity, **x} for x in curves])
                    events.append({**identity, **scores})
                    event_cases.append(pd.DataFrame({**identity, 'target_index': indices, **cases}))
                    provenance.append({**contract, 'selected_checkpoint': complete['selected_checkpoint'],
                                       'cache_sha256': file_hash(path)})
                del model, selected
    return (pd.DataFrame(distributions), pd.DataFrame(events), pd.concat(event_cases, ignore_index=True),
            pd.DataFrame(recording_info), provenance)

distribution_rows, event_rows, event_case_rows, recording_info, analysis_provenance = run_population_analysis()
require(len(event_rows) == 36 and not event_rows.duplicated(['session', 'seed', 'nfe']).any(),
        'The new analysis is incomplete.')
display(recording_info)
event_summary, event_seed_means = summarize(event_rows, ['nfe'],
    ['brier_joint', 'brier_shuffled', 'brier_difference', 'observed_event_rate'])
display(event_summary)
print('Population analysis complete. The scientific figure follows.')


## Main Figure 1 — forecasting neural population activity

The aligned heatmap and next-bin trace use the fixed example from the formal protocol. Every neuron is shown in its stored order. The 90% band is a **predictive interval**, not a confidence interval. Each prediction uses the preceding observed history, so this is not an autonomous ten-second trajectory. The dashed threshold is determined from training data.

The event panel shows all six recordings, with 1 and 16 NFE side by side within each recording. Recording points average three runs. The final row gives the equal-recording mean and one SD across the three seed averages. Positive differences favor the original joint predictions. This control tests predictive dependence and does not establish neuronal connectivity.

In [ ]:
# Match the sans-serif style of the scRNA paper figures.
STYLE = {'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans'],
         'mathtext.fontset': 'dejavusans', 'font.size': 10, 'axes.titlesize': 12,
         'axes.titleweight': 'normal', 'axes.labelsize': 10, 'xtick.labelsize': 10,
         'ytick.labelsize': 10, 'legend.fontsize': 8, 'figure.titlesize': 12,
         'axes.spines.top': True, 'axes.spines.right': True, 'axes.linewidth': .8,
         'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none'}
COLORS = {1: '#0072B2', 16: '#D55E00'}
MARKERS = {1: 'o', 16: '^'}
METHOD_STYLE = {('cfm', 'map'): ('Count Flow Map', '#0072B2', 'o'),
                ('fm', 'unit'): ('Count-FM (unit jump)', '#D55E00', 's'),
                ('fm', 'tau'): ('Count-FM (binomial tau-leap)', '#009E73', '^')}

def panel_title(ax, letter, text):
    ax.set_title(f'{letter}  {text}', loc='center', pad=10)
    ax.set_axisbelow(True)

def export_figure(fig, name):
    directory = PAPER / 'figures'
    directory.mkdir(parents=True, exist_ok=True)
    with plt.rc_context(STYLE):
        for extension in ['pdf', 'svg', 'png']:
            fig.savefig(directory / f'{name}.{extension}', dpi=300, facecolor='white')

figure_session = SpikeSession(ROOT / config['data_root'], e['figure_session'], config['history_bins'])
validate_session(figure_session)
trace_dir = SOURCE / 'evaluations' / f'session_{e["figure_session"]:03d}' / f'seed_{e["figure_seed"]}' / 'cfm'
trace_path = trace_dir / 'trace_map_1.pt'
trace = load_checkpoint(trace_path)
trace_complete = json.loads((trace_dir / 'complete.json').read_text())
trace_indices = figure_session.indices('test')[:e['trace_bins']]
require(trace['signature'] == trace_complete['signature'] and np.array_equal(trace['indices'], trace_indices),
        'The next-bin example is not the fixed trace from the formal experiment.')
trace_values = np.asarray(trace['values'], dtype=float)
trace_counts = np.asarray(figure_session.counts[:, trace_indices])
require(trace_values.shape == (len(trace_indices), 4) and np.isfinite(trace_values).all()
        and np.allclose(trace_values[:, 0], trace_counts.mean(axis=0), rtol=1e-6, atol=1e-8)
        and (trace_values[:, 2] <= trace_values[:, 3]).all(), 'Invalid stored next-bin trace.')
trace_threshold = float(recording_info.set_index('session').loc[e['figure_session'], 'event_threshold_mean_count'])
trace_time = (np.arange(len(trace_indices)) + .5) * figure_session.bin_width_ms / 1000
trace_duration = len(trace_indices) * figure_session.bin_width_ms / 1000
trace_frame = pd.DataFrame({'target_index': trace_indices, 'time_s': trace_time,
                           'observed_mean_count': trace_values[:, 0], 'forecast_mean': trace_values[:, 1],
                           'forecast_q05': trace_values[:, 2], 'forecast_q95': trace_values[:, 3]})
event_recordings = event_rows.groupby(['session', 'nfe'], sort=True).brier_difference.mean().reset_index()

from matplotlib.colors import PowerNorm
from matplotlib.ticker import MaxNLocator, ScalarFormatter
with plt.rc_context(STYLE):
    fig_science = plt.figure(figsize=(10, 5.7))
    outer = fig_science.add_gridspec(1, 2, width_ratios=[2.15, 1], left=.08, right=.97,
                                    bottom=.20, top=.86, wspace=.40)
    left = outer[0].subgridspec(2, 2, width_ratios=[1, .027], height_ratios=[1, 1.55],
                              hspace=.49, wspace=.07)
    heat = fig_science.add_subplot(left[0, 0])
    color_ax = fig_science.add_subplot(left[0, 1])
    forecast_ax = fig_science.add_subplot(left[1, 0], sharex=heat)
    event_ax = fig_science.add_subplot(outer[1])
    im = heat.imshow(trace_counts, origin='upper', aspect='auto', interpolation='nearest',
                     extent=(0, trace_duration, figure_session.dim+.5, .5), cmap='Greys',
                     norm=PowerNorm(.5, vmin=0, vmax=max(1, int(trace_counts.max()))), rasterized=True)
    bar = fig_science.colorbar(im, cax=color_ax)
    bar.locator = MaxNLocator(nbins=3, integer=True)
    bar.update_ticks()
    bar.ax.tick_params(labelsize=8)
    bar.ax.set_title('Count', fontsize=8, pad=5)
    heat.set(ylabel='Neuron', xlim=(0, trace_duration))
    heat.tick_params(axis='x', labelbottom=False)
    panel_title(heat, 'A', 'Observed neural activity')
    forecast_ax.fill_between(trace_time, trace_values[:, 2], trace_values[:, 3],
                             color=COLORS[1], alpha=.20, lw=0, label='90% predictive interval')
    forecast_ax.plot(trace_time, trace_values[:, 1], color=COLORS[1], lw=1.2, label='Forecast mean')
    forecast_ax.plot(trace_time, trace_values[:, 0], color='.15', lw=.85, label='Observed')
    forecast_ax.axhline(trace_threshold, color='#CC79A7', ls='--', lw=1, label='High-activity threshold')
    forecast_ax.set(xlabel='Time in test segment (s)', ylabel='Mean count per neuron')
    forecast_ax.grid(axis='y', alpha=.15)
    forecast_ax.legend(loc='upper center', bbox_to_anchor=(.5, -.28), ncol=2, frameon=False, fontsize=8)
    panel_title(forecast_ax, 'B', 'Next-bin population forecast · 1 NFE')
    event_ax.axvline(0, color='.5', ls='--', lw=1)
    positions = np.arange(len(config['sessions']))
    for offset, nfe in [(-.12, 1), (.12, 16)]:
        values = event_recordings[event_recordings.nfe == nfe].set_index('session').loc[config['sessions'], 'brier_difference'].to_numpy()*1000
        event_ax.scatter(values, positions+offset, color=COLORS[nfe], marker=MARKERS[nfe],
                         s=35, label=f'{nfe} NFE', zorder=3)
        overall = event_summary.set_index('nfe').loc[nfe]
        event_ax.errorbar(overall.brier_difference_mean*1000, len(positions)+.45+offset,
                          xerr=overall.brier_difference_std*1000, fmt=MARKERS[nfe],
                          color=COLORS[nfe], ms=6, capsize=3, lw=1.3, zorder=3)
    event_ax.axhline(len(positions)-.35, color='.85', lw=.8)
    event_ax.set(yticks=[*positions, len(positions)+.45],
                 yticklabels=[f'{s:03d}' for s in config['sessions']]+['Mean'],
                 ylim=(len(positions)+1.05, -1.30), ylabel='Recording',
                 xlabel='Brier score change\n'+r'after shuffling ($\times 10^{-3}$)')
    event_ax.margins(x=.18)
    event_ax.legend(loc='upper right', frameon=False)
    panel_title(event_ax, 'C', 'High-activity events')
    # fig_science.suptitle('Forecasting neural population activity', x=.5, y=.98, ha='center')
    export_figure(fig_science, 'paper_neural_scientific')
    plt.show()


## Run or resume the refined NFE evaluation

This is the potentially long cell. The original grid supplies 14 operating points per recording/run pair. Eleven missing points per pair are added, giving **198 new score evaluations** across all 18 pairs. No training occurs. Progress is printed and scores are saved every 256 histories. All 450 timing measurements cover the 25 operating points and 18 pairs with a single consistent runtime profile. Later runs reuse those timings.

Quality scores already computed by the formal experiment stay unchanged. New scores call the same forecasting and scoring functions with the same test histories, seed, draw count and batch size. Timing is independent of the score RNG. The original evaluation configuration is never edited.

In [ ]:
sweep_rows, sweep_provenance = run_nfe_sweep()
print('Finished the full NFE grid and consistent timing measurements.')


## Appendix table and Main Figure 2 — quality and computational cost

The table contains all 25 operating points, including Count-FM at 128 and 256 NFE. Main Figure 2 uses the common 1–64 NFE grid. Left shows quality versus step budget, right quality versus measured runtime. The same population CRPS appears on both vertical axes. Lines connect actual points in NFE order, without smoothing or extrapolation. Bands and vertical error bars show one seed SD. Horizontal error bars show one seed SD of latency. Numbers on the runtime panel mark 1 and 64 NFE.

The additional high-budget Count-FM rows remain in the table and energy-score diagnostic. No unfavorable recording or point is filtered out.

In [ ]:
sweep_summary, sweep_seed_means = summarize(sweep_rows, ['method', 'sampler', 'nfe'], SCORES)
order = pd.MultiIndex.from_tuples(VARIANTS, names=['method', 'sampler', 'nfe'])
sweep_summary = sweep_summary.set_index(['method', 'sampler', 'nfe']).loc[order].reset_index()
expanded_table = pd.DataFrame({'Method': [METHOD_STYLE[(r.method, r.sampler)][0] for r in sweep_summary.itertuples()],
                               'NFE': sweep_summary.nfe})
for metric, heading in zip(SCORES, HEADINGS):
    digits = DIGITS[metric]
    expanded_table[heading] = [f'{r[metric+"_mean"]:.{digits}f} ± {r[metric+"_std"]:.{digits}f}'
                               for _, r in sweep_summary.iterrows()]
display(expanded_table)

def score_line(ax, part, metric, label, color, marker, xkey='nfe'):
    part = part.sort_values('nfe')
    x = part[xkey].to_numpy(float)
    mean, sd = part[metric+'_mean'].to_numpy(), part[metric+'_std'].to_numpy()
    ax.plot(x, mean, marker=marker, color=color, ms=4, lw=1.4, label=label)
    if xkey == 'nfe':
        ax.fill_between(x, mean-sd, mean+sd, color=color, alpha=.13, lw=0)
    else:
        ax.errorbar(x, mean, yerr=sd, xerr=part.ensemble_ms_std.to_numpy(),
                    fmt='none', ecolor=color, elinewidth=.8, capsize=2, alpha=.65)

with plt.rc_context(STYLE):
    fig_efficiency, axes = plt.subplots(1, 2, figsize=(10, 4.25), sharey=True)
    fig_efficiency.subplots_adjust(left=.085, right=.975, top=.78, bottom=.26, wspace=.23)
    for (method, sampler), (label, color, marker) in METHOD_STYLE.items():
        part = sweep_summary[(sweep_summary.method == method) & (sweep_summary.sampler == sampler)
                             & sweep_summary.nfe.isin(COMMON_NFE)]
        score_line(axes[0], part, 'population_crps', label, color, marker)
        score_line(axes[1], part, 'population_crps', label, color, marker, 'ensemble_ms_mean')
        # Label fixed endpoint budgets; every intermediate evaluated budget remains visible.
        for budget in [1, 64]:
            row = part.set_index('nfe').loc[budget]
            offset = {'cfm': (5, 7), 'unit': (5, -13), 'tau': (5, 7)}[method if method == 'cfm' else sampler]
            axes[1].annotate(str(budget), (row.ensemble_ms_mean, row.population_crps_mean),
                             xytext=offset, textcoords='offset points', color=color, fontsize=8)
    axes[0].set_xscale('log', base=2)
    axes[0].set(xticks=COMMON_NFE, xlabel='Network function evaluations (NFE)', ylabel='Population CRPS ↓')
    axes[0].xaxis.set_major_formatter(ScalarFormatter())
    axes[1].set_xscale('log')
    axes[1].set(xlabel='Latency per 64-draw forecast (ms, log scale)')
    axes[1].margins(x=.20)
    for ax in axes:
        ax.grid(axis='y', alpha=.18)
        ax.margins(y=.17)
    panel_title(axes[0], 'A', 'Quality versus step budget')
    panel_title(axes[1], 'B', 'Quality versus runtime')
    handles, labels = axes[0].get_legend_handles_labels()
    fig_efficiency.legend(handles, labels, loc='lower center', bbox_to_anchor=(.5, .035), ncol=3, frameon=False)
    # fig_efficiency.suptitle('Neural forecast quality and computational cost', x=.5, y=.97, ha='center')
    export_figure(fig_efficiency, 'paper_neural_efficiency')
    plt.show()


## Appendix figure — joint quality, calibration and autonomous forecasting

Energy score uses the full evaluated grid. Calibration and autonomous rollout panels reuse the completed formal Count Flow Map results at 1 and 16 NFE. Coverage uses randomized ranks to account for ties and finite ensembles. Rollouts condition recursively on generated counts. NFE applies per 50-ms transition, so a 500-ms rollout uses ten transitions. The fixed rollout subset has up to 256 histories per recording, compared with up to 4,096 in the next-bin evaluation.

In [ ]:
cfm = flow[flow.method == 'cfm'].copy()
calibration, calibration_seeds = summarize(cfm, ['nfe'], COVERAGE)
roll = rollouts[rollouts.method == 'cfm'].copy()
expected_roll = {(s, seed, nfe, h) for s, seed in PAIRS for nfe in [1, 16] for h in e['rollout_horizons']}
require(not roll.duplicated(['session', 'seed', 'nfe', 'horizon_bins']).any()
        and set(roll[['session', 'seed', 'nfe', 'horizon_bins']].itertuples(index=False, name=None)) == expected_roll,
        'The formal Count Flow Map rollout grid is incomplete.')
require(roll.sampler.eq('map').all() and roll.completed_steps.eq(config['training']['steps']).all()
        and np.isfinite(roll[['energy_score', 'population_crps']]).all().all()
        and np.allclose(roll.horizon_ms, roll.horizon_bins*50)
        and np.allclose(roll.total_nfe, roll.nfe*roll.horizon_bins), 'Invalid rollout values or labels.')
joined = roll.merge(cfm[KEY + ['candidate_index', 'selected_update']], on=KEY, suffixes=('_roll', '_next'), validate='many_to_one')
require(joined.candidate_index_roll.eq(joined.candidate_index_next).all()
        and joined.selected_update_roll.eq(joined.selected_update_next).all(), 'Rollouts use different selected weights.')
roll_summary, roll_seed_means = summarize(roll, ['nfe', 'horizon_ms'], ['energy_score', 'population_crps'])

with plt.rc_context(STYLE):
    fig_appendix, axes = plt.subplots(2, 2, figsize=(10, 7.5))
    fig_appendix.subplots_adjust(left=.085, right=.975, bottom=.09, top=.88, hspace=.52, wspace=.30)
    for (method, sampler), (label, color, marker) in METHOD_STYLE.items():
        part = sweep_summary[(sweep_summary.method == method) & (sweep_summary.sampler == sampler)]
        score_line(axes[0, 0], part, 'energy_score', label, color, marker)
    axes[0, 0].set_xscale('log', base=2)
    axes[0, 0].set(xticks=[1, 4, 16, 64, 256], xlabel='NFE', ylabel='Energy score ↓')
    axes[0, 0].xaxis.set_major_formatter(ScalarFormatter())
    axes[0, 0].legend(frameon=False, fontsize=7, loc='best')
    panel_title(axes[0, 0], 'A', 'Joint forecast quality')
    axes[0, 1].plot([.45, 1], [.45, 1], color='.4', ls='--', lw=1)
    for nfe in [1, 16]:
        row = calibration.set_index('nfe').loc[nfe]
        mean = row[[f'{x}_mean' for x in COVERAGE]].to_numpy(float)
        sd = row[[f'{x}_std' for x in COVERAGE]].to_numpy(float)
        axes[0, 1].plot(np.array(LEVELS)/100, mean, color=COLORS[nfe], marker=MARKERS[nfe], ms=4,
                        lw=1.4, label=f'Count Flow Map · {nfe} NFE')
        axes[0, 1].fill_between(np.array(LEVELS)/100, mean-sd, mean+sd, color=COLORS[nfe], alpha=.13, lw=0)
        part = roll_summary[roll_summary.nfe == nfe].sort_values('horizon_ms')
        for ax, metric in zip(axes[1], ['energy_score', 'population_crps']):
            x, mean, sd = part.horizon_ms.to_numpy(), part[metric+'_mean'].to_numpy(), part[metric+'_std'].to_numpy()
            ax.plot(x, mean, color=COLORS[nfe], marker=MARKERS[nfe], ms=4, lw=1.4)
            ax.fill_between(x, mean-sd, mean+sd, color=COLORS[nfe], alpha=.13, lw=0)
    axes[0, 1].set(xlabel='Nominal coverage', ylabel='Empirical rank coverage', xlim=(.46, 1), ylim=(0, 1.02))
    axes[0, 1].legend(frameon=False, loc='upper left')
    panel_title(axes[0, 1], 'B', 'Population calibration')
    for ax, letter, ylabel in zip(axes[1], ['C', 'D'], ['Energy score ↓', 'Population CRPS ↓']):
        ax.set(xlabel='Forecast horizon (ms)', ylabel=ylabel, xticks=[50, 100, 250, 500])
        panel_title(ax, letter, 'Autonomous population forecasts' if letter == 'D' else 'Autonomous joint forecasts')
    for ax in axes.flat:
        ax.grid(axis='y', alpha=.15)
    # fig_appendix.suptitle('Neural forecasting diagnostics', x=.5, y=.97, ha='center')
    export_figure(fig_appendix, 'paper_neural_diagnostics')
    plt.show()


## Export numeric results and manuscript-ready files

Each figure is saved as PDF, SVG and 300-dpi PNG when displayed. This final cell writes the complete table as editable LaTeX and CSV, the figure captions and LaTeX snippets, underlying summaries, the saved next-bin trace, and a provenance manifest. It marks the export complete only after every recording, run and budget is present. Copy the table's actual LaTeX into the manuscript if you prefer to edit its layout directly. The table uses `booktabs`; figure snippets use `graphicx`.

In [ ]:
SCIENCE_CAPTION = (
    r'Forecasting neural population activity. (A) Observed 50-ms spike counts for every recorded neuron in the fixed example recording. '
    r'The grayscale uses square-root scaling. (B) Observed mean count per neuron, the Count Flow Map predictive mean, and the 5th--95th percentile predictive interval at 1 NFE. '
    r'Each forecast conditions on the preceding 500 ms of observed counts. The 10-s display is a sequence of one-bin-ahead predictions, not a 10-s autonomous forecast. '
    r'The dashed threshold is the 95th percentile of training population activity. This illustration uses one run, with the recording and window fixed by the original evaluation protocol. '
    r'(C) Change in high-activity Brier score after independently permuting predictive draws across neurons within each history, preserving each empirical neuronal marginal. '
    r'Events strictly exceed the recording-specific training threshold. Positive changes favor the original joint forecasts. Each recording point averages three training runs. '
    r'The final row averages six recordings equally and shows one sample SD across the three seed averages. All recordings and both signs are retained.'
)
EFFICIENCY_CAPTION = (
    r'Neural forecast quality and computational cost. Population CRPS evaluates mean counts per neuron and is shown against (A) NFE and (B) measured latency per ensemble of 64 joint forecasts. '
    r'All methods use NFE $1,2,4,8,16,32,64$, with selected weights fixed across budgets. Count-FM samplers share the same selected rate network. '
    r'Lines connect evaluated budgets in increasing NFE order. Point labels in (B) identify 1 and 64 NFE. '
    r'Recordings are averaged equally within each seed. Curves show means over three seed averages, bands and error bars show one sample SD, and horizontal error bars describe latency variation. '
    r'Every latency is measured in the same runtime environment using the original timing protocol, including history encoding and device transfers. '
    r'The appendix table also retains the previously evaluated Count-FM budgets of 128 and 256 NFE. No test result selects a checkpoint or an inference budget.'
)
DIAGNOSTICS_CAPTION = (
    r'Neural forecasting diagnostics. (A) Energy score across the full evaluated step grid, with Euclidean distances normalized by $\sqrt{D}$ for a recording with $D$ neurons. '
    r'(B) Count Flow Map randomized-rank population coverage at 1 and 16 NFE. The diagonal marks nominal coverage. '
    r'(C--D) Autonomous forecasts recursively append generated counts to the history. NFE applies to each 50-ms transition. '
    r'Lines average six recordings equally within each training seed and then average the three seeds. Bands show one sample SD across seed averages. '
    r'Calibration and rollouts reuse the formal evaluation. Rollouts use a separate fixed subset of up to 256 histories per recording, compared with up to 4,096 next-bin histories.'
)
TABLE_CAPTION = (
    r'Neural forecasting within the Count-FM family across the full inference-budget sweep. Each method predicts the next 50-ms count vector from 500 ms of observed history. '
    r'Entries are mean $\pm$ sample SD across three training-seed averages, giving the six recordings equal weight within each seed. '
    r'This SD measures variation across training runs, not uncertainty across biological replicates. '
    r'Energy score normalizes Euclidean distances by $\sqrt{D}$. Population CRPS evaluates mean counts per neuron. RMSE uses the existing finite-ensemble correction. '
    r'Latency is remeasured for every row in one runtime environment and includes generating 64 joint draws, history encoding, and device transfers. '
    r'Previously computed quality scores are reused, and missing budgets evaluate the same frozen selected checkpoints without training or reselection.'
)

def expanded_table_tex():
    lines = [r'\begin{table}[t]', r'\centering', r'\small', r'\setlength{\tabcolsep}{4pt}',
             r'\begin{tabular}{rcccc}', r'\toprule',
             r'NFE & Energy $\downarrow$ & Pop. CRPS $\downarrow$ & RMSE $\downarrow$ & Latency (ms) $\downarrow$ \\']
    for (method, sampler), (label, _, _) in METHOD_STYLE.items():
        lines += [r'\midrule', r'\multicolumn{5}{l}{' + label.replace('tau-leap', r'$\tau$-leap') + r'} \\']
        part = sweep_summary[(sweep_summary.method == method) & (sweep_summary.sampler == sampler)].sort_values('nfe')
        for _, row in part.iterrows():
            values = [f'${row[m+"_mean"]:.{DIGITS[m]}f} \\pm {row[m+"_std"]:.{DIGITS[m]}f}$' for m in SCORES]
            lines.append(' & '.join([str(int(row.nfe)), *values]) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\caption{' + TABLE_CAPTION + '}',
              r'\label{tab:neural-full-nfe}', r'\end{table}']
    return '\n'.join(lines)+'\n'

def figure_tex(name, caption, label):
    return ('\\begin{figure}[t]\n\\centering\n\\includegraphics[width=\\linewidth]{figures/'+name+'.pdf}\n'
            '\\caption{'+caption+'}\n\\label{'+label+'}\n\\end{figure}\n')

require(all(file_hash(SOURCE / name) == value for name, value in INPUT_HASHES.items()),
        'Formal source files changed while this notebook was running. Rerun from the beginning.')
PAPER.mkdir(parents=True, exist_ok=True)
for name, frame in {
    'full_nfe_by_run': sweep_rows, 'full_nfe_summary': sweep_summary, 'full_nfe_seed_means': sweep_seed_means,
    'paper_neural_full_nfe_table': expanded_table,
    'next_bin_trace': trace_frame, 'high_activity_by_run': event_rows,
    'high_activity_by_recording': event_recordings, 'high_activity_summary': event_summary,
    'high_activity_seed_means': event_seed_means, 'recording_thresholds': recording_info,
    'calibration_summary': calibration, 'autonomous_rollout_summary': roll_summary}.items():
    frame.to_csv(PAPER / (name+'.csv'), index=False)
(PAPER / 'paper_neural_full_nfe_table.tex').write_text(expanded_table_tex(), encoding='utf-8')
for name, caption, label in [
    ('paper_neural_scientific', SCIENCE_CAPTION, 'fig:neural-scientific'),
    ('paper_neural_efficiency', EFFICIENCY_CAPTION, 'fig:neural-efficiency'),
    ('paper_neural_diagnostics', DIAGNOSTICS_CAPTION, 'fig:neural-diagnostics')]:
    (PAPER / (name+'.tex')).write_text(figure_tex(name, caption, label), encoding='utf-8')
(PAPER / 'figure_captions.txt').write_text('\n\n'.join([SCIENCE_CAPTION, EFFICIENCY_CAPTION, DIAGNOSTICS_CAPTION]), encoding='utf-8')
manifest = {'purpose': 'paper', 'status': 'complete', 'protocol': protocol, 'source_hashes': INPUT_HASHES,
            'training_performed': False, 'checkpoint_selection_performed': False,
            'common_nfe_grid': COMMON_NFE, 'additional_count_fm_budgets': FM_EXTRA_NFE,
            'sweep_version': SWEEP_VERSION, 'score_implementation_sha256': SCORE_IMPLEMENTATION,
            'timing_runtime': TIMING_RUNTIME, 'timing_profile_id': TIMING_ID,
            'aggregation': 'equal recording means within each seed, then mean and sample SD across three seeds',
            'trace_file': str(trace_path), 'trace_sha256': file_hash(trace_path),
            'fixed_example': {'recording': e['figure_session'], 'training_seed': e['figure_seed'],
                              'nfe': 1, 'target_indices': trace_indices.tolist()},
            'population_checkpoints_and_caches': analysis_provenance, 'sweep_checkpoints_and_caches': sweep_provenance}
atomic_json(manifest, PAPER / 'paper_results_manifest.json')
print('Complete. Paper figures, the 25-row table, captions and numeric summaries are in:')
print(PAPER)
print('Rerun this notebook to reuse completed population, NFE and timing caches.')


## Files to use in the paper

- Main scientific figure — `figures/paper_neural_scientific.pdf`
- Main computational figure — `figures/paper_neural_efficiency.pdf`
- Appendix table — `paper_neural_full_nfe_table.tex`
- Appendix diagnostic figure — `figures/paper_neural_diagnostics.pdf`

The figures share the scRNA sans-serif style. Exact run identifiers appear in provenance, not figure titles. Present the curves as measured, including any nonmonotonicity. The event comparison measures the predictive contribution of generated dependence, while the calibration panel assesses uncertainty accuracy. Neither alone establishes recovery of biological interactions.